# Kafka, first steps

Twenty minutes, one topic, and the four facts that the rest of the Kafka lecture
assumes you have seen: **a topic is a set of partitions**, **the key decides the
partition**, **a consumer group splits the partitions between its members**, and
**a group is a cursor, not a queue**.

Every section asks you to **predict before you run**. The predictions are the point;
the outputs only settle them. Nothing here needs Spark — this is Kafka on its own,
which is the only way to see the parts that Spark and ksqlDB hide from you.

### The client is already installed

There is no `pip` cell in this module: `docker-compose.yml` mounts a script into the
notebook image's `before-notebook.d/`, so `confluent-kafka` is installed when the
container starts. The cell below is a check. If it fails, one line on your host:

```
docker exec -it kafka-first-steps-notebook pip install confluent-kafka==2.5.0
```

In [1]:
import confluent_kafka
print("confluent-kafka", confluent_kafka.version()[0])

confluent-kafka 2.5.0


In [2]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer, Consumer, TopicPartition
import time, collections

BOOTSTRAP = "kafka:9092"      # the broker, inside the docker network
TOPIC     = "elevator-doors"

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
print("broker reachable, topics now:", sorted(admin.list_topics(timeout=10).topics))

broker reachable, topics now: ['__consumer_offsets']


---
## 1) A topic is a set of partitions

Three partitions, one replica each — one broker, so there is nowhere to put a second
copy. In production the replication factor is 3 and the replicas sit on different
brokers; everything below would read the same.

**Why three and not one?** Because a partition is the unit of both ordering and
parallelism: order is guaranteed *inside* a partition and nowhere else, and a consumer
group can never have more working members than the topic has partitions. Choosing the
partition count is choosing the maximum parallelism, before a single message exists.

In [3]:
fs = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)])
for t, f in fs.items():
    f.result()          # raises if the creation failed
    print("created", t)

# the broker needs a moment to publish the new metadata
for _ in range(10):
    meta = admin.list_topics(timeout=10).topics
    if TOPIC in meta and len(meta[TOPIC].partitions) == 3:
        break
    time.sleep(0.5)

for pid, p in sorted(meta[TOPIC].partitions.items()):
    print(f"partition {pid}: leader=broker {p.leader}, replicas={p.replicas}")

created elevator-doors
partition 0: leader=broker 1, replicas=[1]
partition 1: leader=broker 1, replicas=[1]
partition 2: leader=broker 1, replicas=[1]


---
## 2) With no key, the client spreads them itself

Nine messages, no key at all.

**Predict first:** how will they be spread over the three partitions? And in which
partition will message number 7 land?

In [4]:
seen = []

def report(err, msg):
    """Delivery callback: the broker tells us where the message actually went."""
    if err:
        print("FAILED:", err)
    else:
        seen.append((msg.key(), msg.value().decode(), msg.partition(), msg.offset()))

producer = Producer({"bootstrap.servers": BOOTSTRAP})
for i in range(9):
    producer.produce(TOPIC, value=f"message {i}", on_delivery=report)
producer.flush()

# the callbacks fire per BATCH, so they arrive grouped by partition; sort by the
# message number to see the order in which the messages were actually produced
seen.sort(key=lambda r: int(r[1].split()[1]))
for key, val, part, off in seen:
    print(f"{val:10s} -> partition {part}, offset {off}")

order = [part for key, val, part, off in seen]
switches = sum(1 for a, b in zip(order, order[1:]) if a != b)
print()
print("partition per message, in production order:", order)
print(f"switches between consecutive messages: {switches} out of {len(order) - 1}")

message 0  -> partition 2, offset 0
message 1  -> partition 0, offset 0
message 2  -> partition 1, offset 0
message 3  -> partition 0, offset 1
message 4  -> partition 2, offset 1
message 5  -> partition 0, offset 2
message 6  -> partition 0, offset 3
message 7  -> partition 0, offset 4
message 8  -> partition 1, offset 1

partition per message, in production order: [2, 0, 1, 0, 2, 0, 0, 0, 1]
switches between consecutive messages: 6 out of 8


**Count the switches before you believe the word *round-robin*.** Strict round-robin
would switch partition on every single message — eight out of eight. You got **zero or
one**: all nine messages in one partition, or in two.

The client's default partitioner is `consistent_random`: the key is hashed, and a
message *without* a key is *"randomly partitioned"*. But random per **what**? Not per
message. `sticky.partitioning.linger.ms` — default **10 ms** — is a *"delay to wait to
assign new sticky partitions for each topic"*, and its stated purpose is that batching
keyless messages together is cheaper than spreading them. Nine messages produced in a
loop take well under 10 ms, so they are one batch, in one partition.

**So predict again:** the same nine messages, 30 ms apart — longer than the sticky
window. How many switches now?

In [5]:
seen.clear()
for i in range(9):
    producer.produce(TOPIC, value=f"slow {i}", on_delivery=report)
    producer.flush()          # close the batch
    time.sleep(0.03)          # longer than sticky.partitioning.linger.ms (10 ms)

seen.sort(key=lambda r: int(r[1].split()[1]))
order = [part for key, val, part, off in seen]
switches = sum(1 for a, b in zip(order, order[1:]) if a != b)
print("partition per message, 30 ms apart:", order)
print(f"switches: {switches} out of {len(order) - 1}")

partition per message, 30 ms apart: [1, 2, 0, 2, 2, 2, 0, 0, 1]
switches: 5 out of 8


Now they move — and not in the orderly 0,1,2,0,1,2 of the folklore, because the choice
is **random**, not rotating. Two consecutive messages can land in the same partition by
chance; over enough of them the spread evens out.

The practical reading is worth more than the mechanism. **A fast producer without keys
concentrates; a slow one spreads.** So a topic that looks perfectly balanced in a test
with a sleep in the loop can arrive skewed in production, where the same code runs flat
out — and the imbalance is not in your data, it is in your throughput.

Two more things this cell shows. The partition and the offset are **not** chosen by the
producer: they come back from the broker, because they are decided when the message is
appended to the log. And the delivery callbacks arrive **grouped by partition**, which is
why both cells sort them — what the client hands back is batches, and reading that order
as the production order is a mistake that is easy to make and hard to notice.

---
## 3) With a key, the partition is a function of the key

Now three elevator units, three messages each, with `unitId` as the key.

**Predict first:** will each unit's three messages land together? Will three keys
land in three different partitions? And what would change if the topic had 6
partitions instead of 3?

In [6]:
seen.clear()
for unit in ("EU-001", "EU-002", "EU-003"):
    for i in range(3):
        producer.produce(TOPIC, key=unit, value=f"{unit} door {i}", on_delivery=report)
producer.flush()

by_key = collections.defaultdict(set)
for key, val, part, off in seen:
    by_key[key.decode()].add(part)

for key in sorted(by_key):
    print(f"{key} -> partition(s) {sorted(by_key[key])}")

EU-001 -> partition(s) [1]
EU-002 -> partition(s) [1]
EU-003 -> partition(s) [2]


**A key never spreads.** Its partition is `hash(key) % partitions`, so every message
for `EU-001` is in one partition, in the order it was produced. That is the whole
reason keys exist: they are how you buy ordering for the things that need it, without
paying for a global order that nothing can scale.

And the relation is **N:1**, which is the answer to that line of the quiz: several
keys share a partition — with three keys and three partitions, two of them may well
have collided above — but one key is never split across two. Adding partitions later
changes `% partitions` and therefore moves keys: repartitioning an existing topic
breaks the ordering guarantee for the keys that move, which is why the partition count
is a decision you would rather get right once.

---
## 4) One consumer in a group reads every partition

A consumer joins a **group**. The group is what the broker tracks: which partitions
its members hold, and how far each has read.

**Predict:** one consumer, three partitions — how many of the 27 messages does it get?

In [7]:
def drain(consumers, seconds=10.0, quiet_for=2.0):
    """Poll every consumer until `quiet_for` seconds pass with nothing new.

    Returns {consumer name: [messages]}. Polling all of them in turn is what lets
    the group rebalance: a member that does not poll is a member that is not there.
    """
    got = {name: [] for name, _ in consumers}
    deadline, last = time.time() + seconds, time.time()
    while time.time() < deadline and time.time() - last < quiet_for:
        for name, c in consumers:
            msg = c.poll(0.2)
            if msg is None or msg.error():
                continue
            got[name].append(msg)
            last = time.time()
    return got

def group(name, gid):
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": gid,
                  "auto.offset.reset": "earliest"})
    c.subscribe([TOPIC])
    return (name, c)

solo = group("consumer-1", "dashboard")
got = drain([solo])
print("assigned partitions:", sorted(p.partition for p in solo[1].assignment()))
print("messages received  :", len(got["consumer-1"]))

assigned partitions: [0, 1, 2]
messages received  : 27


---
## 5) Two consumers in the same group split the partitions

A second consumer joins the **same** group. The broker rebalances.

**Predict:** how do three partitions divide between two consumers? And what happens
to a **fourth** consumer in a group with three partitions?

In [8]:
second = group("consumer-2", "dashboard")

# both must poll for the rebalance to complete; there is nothing to read, so the
# only thing this drain does is let the group settle
drain([solo, second], seconds=20.0, quiet_for=8.0)

for name, c in (solo, second):
    print(f"{name}: partitions {sorted(p.partition for p in c.assignment())}")

consumer-1: partitions [2]
consumer-2: partitions [0, 1]


Two and one, or one and two — which member gets two partitions is an accident of the
assignment strategy, and nothing in your design should depend on it.

What *is* guaranteed: **inside one group, a partition has exactly one reader.** So a
fourth consumer in this group would sit idle, holding nothing, consuming nothing — the
parallelism of a consumer group is capped by the partition count of the topic, which is
the same decision as in section 1, met again from the other end. Add it yourself and
look, it takes one line.

This is also the relation the quiz asks for: consumer to partition is **1:N** — one
consumer may hold several partitions, a partition is held by exactly one consumer.

---
## 6) A group is a cursor, not a queue

Reading does not consume. The messages are still in the log — that is what retention
is for. What a group holds is a **position**.

**Predict:** a brand-new group subscribes with `auto.offset.reset=earliest`. How many
of the 27 messages does it see?

In [9]:
audit = group("audit", "audit-trail")
got = drain([audit])
print("the new group received:", len(got["audit"]), "messages")
print()

print("where each group stands, as the BROKER remembers it:")
for name, c in (solo, second, audit):
    for tp in sorted(c.assignment(), key=lambda t: t.partition):
        lo, hi = c.get_watermark_offsets(tp, timeout=5)
        com = c.committed([tp], timeout=5)[0].offset
        com = "nothing committed" if com < 0 else f"committed {com}"
        print(f"  {name:10s} partition {tp.partition}: {com:20s} "
              f"log holds offsets {lo}..{hi - 1}")

the new group received: 27 messages

where each group stands, as the BROKER remembers it:
  consumer-1 partition 2: committed 9          log holds offsets 0..8
  consumer-2 partition 0: committed 8          log holds offsets 0..7
  consumer-2 partition 1: committed 10         log holds offsets 0..9
  audit      partition 0: nothing committed    log holds offsets 0..7
  audit      partition 1: nothing committed    log holds offsets 0..9
  audit      partition 2: nothing committed    log holds offsets 0..8


Read the *log holds* column first: the messages are **all still there**. Nothing was
removed by being read — they leave when retention says so, not when a consumer is done
with them.

The committed offset is the group's bookmark, and it lives on the broker rather than in
the consumer, which is what lets a crashed member be replaced by another that picks up
where it left off. A group may show *nothing committed*: committing is periodic and a
member that has just been through a rebalance may not have reached its next commit —
which is worth seeing too, because "the consumer read it" and "the group has recorded
that it read it" are different statements, and only the second one survives a crash.

Two groups, the same messages, two independent positions. This is why a Kafka topic
can feed a dashboard, a batch export and a machine-learning pipeline at once without
any of them knowing about the others — and why "the consumer is slow" is a question
about **lag**, the distance between a position and the end of the log, rather than
about a queue filling up.

It is also the mechanism behind `startingOffsets: "earliest"` in the Spark modules of
the next two lectures: Spark is a consumer group like any other.

---
## 7) The same event, in Avro

Five minutes, and it changes a number you are about to be asked for. The assignment at
the end of this lecture has you size a fleet of 150,000 units in MB/s, taking a JSON
event at about 250 bytes. **The encoding is a factor in that product**, and it is the
one you control without buying anything.

[Apache Avro](https://avro.apache.org/docs/) is a data serialization system: a schema
in JSON, and a compact binary encoding that carries **no field names at all** — the
reader knows them from the schema. The schema for our door event is next to this
notebook, in `door-event.avsc`. You exchanged it by hand, which is the honest way to
start: no registry, no infrastructure, a file.

In [10]:
import json

schema_str = open("door-event.avsc").read()
print(schema_str)

{
  "namespace": "it.polimi.sda",
  "type": "record",
  "name": "DoorEvent",
  "doc": "One door movement of one elevator. The naive translation of the JSON event: every field a string, exactly as it was written by hand.",
  "fields": [
    {"name": "event",     "type": "string"},
    {"name": "unitId",    "type": "string"},
    {"name": "ts",        "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "car",       "type": "string"},
    {"name": "floor",     "type": "int"},
    {"name": "servedDir", "type": "string"}
  ]
}



**Predict:** the same event as JSON is 115 bytes. How small in Avro?

In [11]:
import io, datetime
from fastavro import parse_schema, schemaless_writer

event = {"event": "DoorOpened", "unitId": "EU-001",
         "ts": datetime.datetime(2026, 10, 15, 8, 0, 0,
                                 tzinfo=datetime.timezone.utc),
         "car": "A", "floor": 3, "servedDir": "UP"}

def avro_bytes(schema_dict, record):
    buf = io.BytesIO()
    schemaless_writer(buf, parse_schema(schema_dict), record)
    return buf.getvalue()

as_json = json.dumps({**event, "ts": "2026-10-15T08:00:00"}).encode()
as_avro = avro_bytes(json.loads(schema_str), event)

print("JSON               :", len(as_json), "bytes")
print("Avro               :", len(as_avro), "bytes ",
      f"-> {len(as_json)/len(as_avro):.1f}x smaller")
print("the schema itself  :", len(schema_str.encode()), "bytes, once, not per message")

JSON               : 115 bytes
Avro               : 30 bytes  -> 3.8x smaller
the schema itself  : 551 bytes, once, not per message


Now change **the schema and nothing else**. `event` and `servedDir` take one of a few
known values, so they are not strings, they are **enums** — and an enum on the wire is
an index, not a word.

**Predict again** before you run it.

In [12]:
better = json.loads(schema_str)
better["fields"][0]["type"] = {"type": "enum", "name": "EventType",
    "symbols": ["DoorOpened", "CarMoved", "HallCall", "Fault"]}
better["fields"][5]["type"] = {"type": "enum", "name": "Dir",
    "symbols": ["UP", "DOWN", "NONE"]}

as_enum = avro_bytes(better, event)
print("Avro, with enums   :", len(as_enum), "bytes ",
      f"-> {len(as_json)/len(as_enum):.1f}x smaller than JSON")

Avro, with enums   : 18 bytes  -> 6.4x smaller than JSON


The field names were already gone; now the *values* are gone too, and what is left on
the wire is almost only the floor number and the unit id. **Every byte you removed
moved into the schema**, which travels once.

Put that back into the assignment: a quarter or a sixth of the MB/s, for the same fleet,
the same rate and the same information. That is not a micro-optimisation, it is a
different cluster.

### Does it also mean more messages per batch?

It should, if a batch is bounded in **bytes**. The client will tell us: librdkafka
publishes statistics, and among them the average batch size, the average number of
messages per batch, and the bytes actually sent to the broker.

A thousand messages of each encoding, `linger.ms` at 50 so the batcher has something to
do — and each one twice: once with the default `batch.size`, once with it set to
**16 KB**, which is the default of the Java client. **Predict what changes between the
two, and what does not.**

In [13]:
for t, f in admin.create_topics([
        NewTopic("avro-demo-json", num_partitions=1, replication_factor=1),
        NewTopic("avro-demo-avro", num_partitions=1, replication_factor=1)]).items():
    f.result()
    print("created", t)

created avro-demo-json
created avro-demo-avro


In [14]:
def measure(topic, payload, batch_size=None, n=1000):
    """Produce n copies of `payload` and read the client's own statistics."""
    stats = {}
    conf = {"bootstrap.servers": BOOTSTRAP,
            "linger.ms": 50,
            "statistics.interval.ms": 500,
            "stats_cb": lambda raw: stats.update(json.loads(raw))}
    if batch_size:
        conf["batch.size"] = batch_size
    p = Producer(conf)
    for _ in range(n):
        p.produce(topic, value=payload)
    p.flush()
    deadline = time.time() + 5          # wait for one statistics sample
    while not stats and time.time() < deadline:
        p.poll(0.2)
    top = stats.get("topics", {}).get(topic, {})
    return (len(payload),
            sum(b.get("txbytes", 0) for b in stats.get("brokers", {}).values()),
            top.get("batchsize", {}).get("avg"),
            top.get("batchcnt", {}).get("avg"))

def fmt(v):
    return "n/a" if v is None else str(int(v))

head = ("encoding / setting", "payload", "wire bytes", "batch bytes", "msg/batch")
print("%-24s %8s %11s %12s %10s" % head)
for enc, topic, payload in (("JSON", "avro-demo-json", as_json),
                            ("Avro", "avro-demo-avro", as_enum)):
    for bs, label in ((None, "batch.size default"), (16384, "batch.size 16 KB")):
        size, wire, b_bytes, b_msgs = measure(topic, payload, bs)
        print("%-24s %8d %11d %12s %10s"
              % (enc + " " + label, size, wire, fmt(b_bytes), fmt(b_msgs)))

encoding / setting        payload  wire bytes  batch bytes  msg/batch
JSON batch.size default       115      125282        41663        333
JSON batch.size 16 KB         115      125644        13891        111
Avro batch.size default        18       26280         8663        333
Avro batch.size 16 KB          18       26332         6497        250


Three readings, and the middle one is the one worth arguing about.

**The bytes fall, but by less than the payload did.** A thousand events of 115 bytes put
about 125 KB on the wire, a thousand of 18 bytes about 26 KB: the payload shrank 6.4x
and the traffic only about 4.8x. The difference is **roughly ten bytes per record** that
belong to the batch format — offsets, timestamps, lengths — and no encoding of yours can
remove them. Below a certain size the envelope costs more than the letter.

**With the default `batch.size` the messages per batch do not change at all** — 333
either way, three batches for a thousand messages. That default is a megabyte, and a
thousand of even the fat messages come nowhere near it, so what closed each batch was
**time**: `linger.ms`, and the `flush()` at the end. A smaller payload cannot pack a
batch that was never full.

**At 16 KB the two part company, and the reason is not the obvious one.** JSON's batches
stop at about 14 KB — the ceiling, near enough — and hold ~111 messages. Avro's stay
around 6 KB, nowhere near the ceiling, and hold ~250: **more than twice as many messages
per request**. So the honest description is not that the smaller payload fills the same
batch better. It is that **the smaller payload never reaches the ceiling at all**, while
the larger one keeps hitting it. Your encoding decides whether your own batching limit is
a limit.

And note what did *not* move: the bytes on the wire are the same under both settings.
Batching changes the number of requests, not the volume — which is why it is a latency
and CPU knob, and the encoding is the bandwidth one.

*(If a figure comes back as `None`, your build of librdkafka does not publish that
particular counter; the bytes are always there.)*

### And now the part this module does not teach

You exchanged the schema **by hand**, and for one producer and one consumer in one
notebook that is entirely fine. Now count what you would have to keep in step in a real
system: every producer, every consumer, every replay of old data, and next week a field
is added. Whose copy of `door-event.avsc` is the truth?

Sending the schema **with** each message answers it and undoes everything you just
measured. The other answer is a **schema registry**: a service that holds the schemas,
hands each one an id, and lets the message carry the id instead — plus a rule about
which changes are allowed, so that a new producer cannot break an old consumer.

That is also why the Avro support inside `confluent-kafka` — `AvroSerializer` — takes a
registry client as its **first argument**: it is built for that world, not for this cell.
Which of the two you need is an architectural decision, and now you have the numbers to
make it.

* [Apache Avro documentation](https://avro.apache.org/docs/) — the schema language and
  the binary encoding
* [Confluent Schema Registry](https://docs.confluent.io/platform/current/schema-registry/index.html)
  — *"a centralized repository for managing and validating schemas for topic message
  data, and for serialization and deserialization of the data over the network"*

---
## Clean up

In [15]:
for name, c in (solo, second, audit):
    c.close()

topics = [TOPIC, "avro-demo-json", "avro-demo-avro"]
for t, f in admin.delete_topics(topics, operation_timeout=30).items():
    f.result()
    print("deleted", t)

deleted elevator-doors
deleted avro-demo-json
deleted avro-demo-avro
